# 203. Activation Steering：怎样从表征方向控制 LLM 行为？

> **面试问题：怎样估计、归一化和注入 steering vector，并以层/位置/版本 gate、剂量曲线和任务保持评测防止副作用？**

## 先给结论

这类题要把论文概念拆为可判定的数学/状态合同：方向从什么对比样本估计、解码如何保留合法候选、熵统计的概率空间是什么、权重编辑如何验证 rewrite/generalization/locality。教学实现用受控小向量，不代表真实模型安全、语言质量或跨领域泛化。

## 一手资料

- [Representation Engineering](https://arxiv.org/abs/2310.01405)
- [Activation Addition](https://arxiv.org/abs/2308.10248)
- [Model Behavior Steering](https://arxiv.org/abs/2308.10248)

In [ ]:
notebook_contract = {"mode": "small-controlled-arrays", "oracle": "assertions", "production": "needs-evaluation-and-versioning"}  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["mode"] == "small-controlled-arrays"  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["oracle"] == "assertions"  # 执行本行的状态、计算或校验逻辑。
assert "versioning" in notebook_contract["production"]  # 执行本行的状态、计算或校验逻辑。
assert len(notebook_contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 问题拆解：steering vector 是条件表征的差方向

最小的 activation steering 不是随便给 hidden state 加噪声，而是从同一任务条件下的正/负示例估计方向。例如“诚实回答”与“编造回答”的中间层表征均值差，必须绑定模型、层、token 位置和采样规则。


In [ ]:
import math  # 执行本行的状态、计算或校验逻辑。
positive = [(3.0, 1.0), (2.0, 2.0)]  # 执行本行的状态、计算或校验逻辑。
negative = [(1.0, 1.0), (0.0, 2.0)]  # 执行本行的状态、计算或校验逻辑。
def mean(vectors):  # 执行本行的状态、计算或校验逻辑。
    return tuple(sum(vector[index] for vector in vectors) / len(vectors) for index in range(len(vectors[0])))  # 执行本行的状态、计算或校验逻辑。
pos_mean = mean(positive)  # 执行本行的状态、计算或校验逻辑。
neg_mean = mean(negative)  # 执行本行的状态、计算或校验逻辑。
direction = tuple(a - b for a, b in zip(pos_mean, neg_mean))  # 执行本行的状态、计算或校验逻辑。
assert pos_mean == (2.5, 1.5)  # 执行本行的状态、计算或校验逻辑。
assert neg_mean == (0.5, 1.5)  # 执行本行的状态、计算或校验逻辑。
assert direction == (2.0, 0.0)  # 执行本行的状态、计算或校验逻辑。


## 2. 归一化：强度和方向必须分离

不归一化时，样本规模或表示尺度会意外改变干预强度。将方向单位化并把 alpha 作为独立超参，才能系统测试强度曲线；零向量必须明确拒绝，不能产生 NaN 后继续生成。


In [ ]:
def normalize(vector):  # 执行本行的状态、计算或校验逻辑。
    norm = math.sqrt(sum(value * value for value in vector))  # 执行本行的状态、计算或校验逻辑。
    if norm == 0:  # 执行本行的状态、计算或校验逻辑。
        raise ValueError("零方向不能用于 steering")  # 执行本行的状态、计算或校验逻辑。
    return tuple(value / norm for value in vector)  # 执行本行的状态、计算或校验逻辑。
unit_direction = normalize(direction)  # 执行本行的状态、计算或校验逻辑。
assert unit_direction == (1.0, 0.0)  # 执行本行的状态、计算或校验逻辑。
assert math.isclose(sum(value * value for value in unit_direction), 1.0)  # 执行本行的状态、计算或校验逻辑。
assert normalize((0.0, 2.0)) == (0.0, 1.0)  # 执行本行的状态、计算或校验逻辑。


## 3. 注入：只在指定层和位置修改 activation

控制面要显式指定 layer/token selector，避免把同一向量加到每一层、所有 token。这里以二维激活模拟一个目标位置；生产实现还要处理 batch、head、残差流 dtype 和 hook 生命周期。


In [ ]:
def steer(hidden, alpha, direction, enabled):  # 执行本行的状态、计算或校验逻辑。
    if not enabled:  # 执行本行的状态、计算或校验逻辑。
        return hidden  # 执行本行的状态、计算或校验逻辑。
    return tuple(value + alpha * delta for value, delta in zip(hidden, direction))  # 执行本行的状态、计算或校验逻辑。
hidden = (0.2, -0.4)  # 执行本行的状态、计算或校验逻辑。
steered = steer(hidden, 0.5, unit_direction, True)  # 执行本行的状态、计算或校验逻辑。
assert steered == (0.7, -0.4)  # 执行本行的状态、计算或校验逻辑。
assert steer(hidden, 0.5, unit_direction, False) == hidden  # 执行本行的状态、计算或校验逻辑。
assert steered[1] == hidden[1]  # 执行本行的状态、计算或校验逻辑。


## 4. 门控：任务、层和安全策略决定能否干预

steering 不应绕过权限或安全策略。示例 gate 限定允许层和经过验证的目标；真实系统还要按模型版本、用户权限、任务域和实验开关审计，禁止把研究 hook 暴露成任意用户输入。


In [ ]:
def allowed_steering(layer, target, policy):  # 执行本行的状态、计算或校验逻辑。
    return layer in policy["layers"] and target in policy["targets"] and policy["enabled"]  # 执行本行的状态、计算或校验逻辑。
policy = {"layers": {12}, "targets": {"honesty"}, "enabled": True}  # 执行本行的状态、计算或校验逻辑。
assert allowed_steering(12, "honesty", policy)  # 执行本行的状态、计算或校验逻辑。
assert not allowed_steering(13, "honesty", policy)  # 执行本行的状态、计算或校验逻辑。
assert not allowed_steering(12, "other", policy)  # 执行本行的状态、计算或校验逻辑。


## 5. 剂量曲线：行为增益与任务保持必须一起测

alpha 过小可能没有效果，过大会破坏原始任务或诱发副作用。教学例子用一个线性行为分数和任务分数展示 Pareto 取舍；真实评估需要盲测、多任务、长输出和红队集。


In [ ]:
def score(alpha):  # 执行本行的状态、计算或校验逻辑。
    behavior = min(1.0, 0.4 + 0.3 * alpha)  # 执行本行的状态、计算或校验逻辑。
    task = max(0.0, 1.0 - 0.15 * alpha)  # 执行本行的状态、计算或校验逻辑。
    return {"behavior": behavior, "task": task}  # 执行本行的状态、计算或校验逻辑。
low, high = score(0.5), score(3.0)  # 执行本行的状态、计算或校验逻辑。
assert high["behavior"] > low["behavior"]  # 执行本行的状态、计算或校验逻辑。
assert high["task"] < low["task"]  # 执行本行的状态、计算或校验逻辑。
assert score(0.0)["task"] == 1.0  # 执行本行的状态、计算或校验逻辑。


## 6. 失败分支：方向漂移和模型版本不匹配要拒绝

某个模型/层/模板上得到的向量不能默认迁移到另一 checkpoint。即使 hidden size 相同，语义也可能改变；因此 steering artifact 应验证模型、层、token selector 与模板指纹。


In [ ]:
def compatible_artifact(artifact, runtime):  # 执行本行的状态、计算或校验逻辑。
    fields = ("model", "layer", "selector", "template")  # 执行本行的状态、计算或校验逻辑。
    return all(artifact[field] == runtime[field] for field in fields)  # 执行本行的状态、计算或校验逻辑。
artifact = {"model": "demo-v1", "layer": 12, "selector": "last-user", "template": "chat-v1"}  # 执行本行的状态、计算或校验逻辑。
runtime = dict(artifact)  # 执行本行的状态、计算或校验逻辑。
assert compatible_artifact(artifact, runtime)  # 执行本行的状态、计算或校验逻辑。
assert not compatible_artifact(artifact, {**runtime, "layer": 13})  # 执行本行的状态、计算或校验逻辑。
assert not compatible_artifact(artifact, {**runtime, "model": "demo-v2"})  # 执行本行的状态、计算或校验逻辑。


## 7. 评测：目标行为、保持能力与副作用分桶报告

单个通过率无法证明控制成功。至少分开报告目标行为成功、原任务正确、拒答/安全回归和不相关任务变化；这里要求每类都有样本，防止平均数掩盖局部失效。


In [ ]:
def bucket_rates(rows):  # 执行本行的状态、计算或校验逻辑。
    result = {}  # 执行本行的状态、计算或校验逻辑。
    for key in ("target", "task", "safety"):  # 执行本行的状态、计算或校验逻辑。
        values = [row[key] for row in rows]  # 执行本行的状态、计算或校验逻辑。
        result[key] = sum(values) / len(values)  # 执行本行的状态、计算或校验逻辑。
    return result  # 执行本行的状态、计算或校验逻辑。
rates = bucket_rates([{"target": 1, "task": 1, "safety": 1}, {"target": 1, "task": 0, "safety": 1}])  # 执行本行的状态、计算或校验逻辑。
assert rates["target"] == 1.0  # 执行本行的状态、计算或校验逻辑。
assert rates["task"] == 0.5  # 执行本行的状态、计算或校验逻辑。
assert rates["safety"] == 1.0  # 执行本行的状态、计算或校验逻辑。


## 8. 制品：方向数据与注入配置必须可复放

保存 direction 数值还不够，还应保存构建样本版本、layer、selector、alpha、模型/template 版本和评测集。生产中需要对 steering 权限、审计日志和回滚开关实施额外治理。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
steering_artifact = {"direction": unit_direction, "alpha": 0.5, "model": "demo-v1", "layer": 12, "dataset": "contrast-v1"}  # 执行本行的状态、计算或校验逻辑。
fingerprint = hashlib.sha256(json.dumps(steering_artifact, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert steering_artifact["alpha"] == 0.5  # 执行本行的状态、计算或校验逻辑。
assert steering_artifact["direction"] == (1.0, 0.0)  # 执行本行的状态、计算或校验逻辑。
assert len(fingerprint) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

回答时先说明目标与状态，再给出核心公式、失败反例、独立评测和版本化制品。不要把对一个合成向量/几个候选的断言通过，误说成真实大模型上已经可靠、无偏或安全。
